# S&P 500 Valuation Analysis Pipeline 📈
**ECM Research | Bulge Bracket Bank**

This notebook automates the extraction, cleaning, and analysis of S&P 500 constituent data. It scrapes live financial metrics from the **Finviz Screener** and historical P/E benchmarks from **Multpl.com**. Finally, it generates a multi-sheet, dynamically formatted Excel dashboard to identify over-valued and under-valued sectors and companies.

### Key Features:
- **Dynamic Historical Benchmarking:** Scrapes 150+ years of historical S&P 500 P/E data to establish a statistically sound median baseline.
- **Robust Scraping:** Includes auto-pagination, exponential backoff, and regex parsing to bypass common anti-bot measures.
- **Automated Formatting:** Exports to an `.xlsx` file complete with Conditional Formatting, Bar Charts, and an Extremes Dashboard.


## Step 1: Scrape Dynamic Historical Benchmark
Instead of hardcoding a 17.5x P/E ratio, we scrape **Multpl.com** for over 150 years of Trailing Twelve Month (TTM) P/E data. 

*Note: We calculate the **Median** rather than the Mean. The Mean is heavily skewed by massive spikes during recessions (e.g., the 2009 crash where P/E mathematically hit 123x as earnings approached zero). The median provides a much more accurate "normal" historical baseline.*


In [ ]:
import argparse
import re
import sys
import json
import time
import random
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("sp500")

# ─────────────────────────────────────────────────────────────
# GLOBAL SESSION & HEADERS
# ─────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
})

# ─────────────────────────────────────────────────────────────
# HISTORICAL S&P 500 P/E BENCHMARK (MULTPL.COM)
# ─────────────────────────────────────────────────────────────
MULTPL_PE_URL = "https://www.multpl.com/s-p-500-pe-ratio/table/by-year"
FALLBACK_HIST_PE   = 17.5

def scrape_sp500_historical_pe() -> float:
    """Scrapes historical S&P 500 P/E from Multpl.com and returns the median value."""
    try:
        headers = session.headers.copy()
        resp = session.get(MULTPL_PE_URL, headers=headers, timeout=15)
        resp.raise_for_status()
        
        # Regex to find all table rows with Date and Value cells safely
        rows = re.findall(r'<tr.*?<td>(.*?)</td>.*?<td>(.*?)</td', resp.text, re.DOTALL)
        
        pe_values = []
        for date_str, val_str in rows:
            if "Date" in date_str or "Value" in date_str:
                continue
                
            # Clean HTML tags, estimate crosses (†), and unicode spaces
            val_clean = re.sub(r'<[^>]+>', '', val_str).strip()
            val_clean = val_clean.replace('†', '').replace('&#x2002;', '').strip()
            val_clean = re.sub(r'[^\d.]', '', val_clean)
            
            if val_clean:
                pe_values.append(float(val_clean))
        
        if not pe_values:
            raise ValueError("No valid numeric P/E values extracted.")
            
        # Calculate the median manually
        pe_values.sort()
        n = len(pe_values)
        if n % 2 == 0:
            median_pe = (pe_values[n//2 - 1] + pe_values[n//2]) / 2.0
        else:
            median_pe = pe_values[n//2]
            
        return round(float(median_pe), 2)

    except Exception as e:
        log.warning(f"Multpl.com scrape failed: {str(e)}. Defaulting to {FALLBACK_HIST_PE}x.")
        return FALLBACK_HIST_PE
  



# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# 1. Fetch historical average safely
HIST_AVG_PE = scrape_sp500_historical_pe()
if HIST_AVG_PE != FALLBACK_HIST_PE:
    log.info(f"Successfully scraped Multpl.com. Historical Median P/E: {HIST_AVG_PE}x")

# 2. Establish High/Low thresholds
PE_HIGH_BOUND = HIST_AVG_PE + 2.5
PE_LOW_BOUND  = HIST_AVG_PE - 2.5

# 3. Finviz Settings
FINVIZ_BASE   = "https://finviz.com/screener.ashx"
FINVIZ_COLS   = "0,1,2,3,4,6,7,8,14,16,77,67,65,66"
FINVIZ_FILTER = "idx_sp500"
PAGE_SIZE     = 20


16:12:15  INFO     Successfully scraped Multpl.com. Historical Median P/E: 15.22x


## Step 2: Live Finviz Screener Scraper
This class connects to Finviz, identifies the total number of companies in the S&P 500, and auto-paginates through the results until all constituent data is downloaded.


In [23]:
# ─────────────────────────────────────────────────────────────
# STEP 1 — FINVIZ SCRAPER
# ─────────────────────────────────────────────────────────────

class FinvizScraper:
    """
    Auto-paginating Finviz screener scraper.
    - Discovers total result count on first page
    - Iterates r=1, r=21, r=41 … until all rows fetched
    - Retries with exponential back-off on network errors
    - Flexible column parsing: reads whatever headers Finviz returns
    """

    def __init__(self, delay: float = 1.2, max_retries: int = 3):
        self.delay = delay
        self.max_retries = max_retries
        self.session = session

    # ── HTTP ──────────────────────────────────────────────────

    def _get(self, params: dict) -> BeautifulSoup:
        for attempt in range(1, self.max_retries + 1):
            try:
                resp = self.session.get(FINVIZ_BASE, params=params, timeout=20)
                resp.raise_for_status()
                return BeautifulSoup(resp.text, "lxml")
            except requests.RequestException as exc:
                wait = attempt * 3 + random.uniform(0, 2)
                log.warning(
                    f"Attempt {attempt}/{self.max_retries} failed ({exc}). "
                    f"Retrying in {wait:.1f}s …"
                )
                time.sleep(wait)
        raise RuntimeError(
            f"Finviz unreachable after {self.max_retries} attempts. "
            "Check network or run with --demo flag for offline testing."
        )

    # ── Parsing ──────────────────────────────────────────────

    @staticmethod
    def _total_count(soup: BeautifulSoup) -> int:
        """Extract 'Total: NNN' from page. Returns 0 if not found."""
        for text in soup.stripped_strings:
            m = re.search(r"Total:\s*(\d+)", text)
            if m:
                return int(m.group(1))
        return 0

    @staticmethod
    def _find_table(soup: BeautifulSoup):
        """Locate the main screener results table robustly."""
        # Try known class names first
        for selector in [
            {"class": "screener_table"},
            {"id": "screener-views-table"},
        ]:
            t = soup.find("table", selector)
            if t:
                return t
        # Fallback: any table that contains screener-link-primary anchors
        for t in soup.find_all("table"):
            if t.find("a", class_="screener-link-primary"):
                return t
        return None

    def _parse_page(self, soup: BeautifulSoup) -> list[dict]:
        table = self._find_table(soup)
        if not table:
            log.warning("Screener table not found on this page — layout may have changed.")
            return []

        all_rows = table.find_all("tr")
        if not all_rows:
            return []

        # First row = headers
        headers = [th.get_text(strip=True) for th in all_rows[0].find_all(["th", "td"])]

        rows = []
        for tr in all_rows[1:]:
            cells = tr.find_all("td")
            if not cells:
                continue
            row = {}
            for i, td in enumerate(cells):
                key = headers[i] if i < len(headers) else f"col_{i}"
                a = td.find("a", class_="screener-link-primary")
                row[key] = a.get_text(strip=True) if a else td.get_text(strip=True)
            # keep only real stock rows (Ticker looks like a ticker)
            ticker_val = row.get("Ticker", row.get("No.", ""))
            if any(c.isalpha() for c in ticker_val):
                rows.append(row)

        return rows

    # ── Public API ──────────────────────────────────────────

    def fetch_sp500(self, max_rows: int | None = None) -> pd.DataFrame:
        log.info("Connecting to Finviz S&P 500 screener …")
        params = dict(v=152, f=FINVIZ_FILTER, ft=4, r=1, c=FINVIZ_COLS)

        # Page 1 — discover total
        soup  = self._get(params)
        total = self._total_count(soup)
        if total == 0:
            log.warning("Could not read total count — will paginate until empty page.")
            total = 600

        log.info(f"Finviz reports {total} S&P 500 constituents.")
        rows = self._parse_page(soup)
        log.info(f"  Page r=1 → {len(rows)} rows")
        time.sleep(self.delay + random.uniform(0, 0.4))

        # Remaining pages
        r = PAGE_SIZE + 1
        while r <= total:
            if max_rows and len(rows) >= max_rows:
                break
            params["r"] = r
            log.info(f"  Fetching rows {r}–{r + PAGE_SIZE - 1} …")
            soup  = self._get(params)
            page  = self._parse_page(soup)
            if not page:
                log.info("  Empty page — pagination complete.")
                break
            rows.extend(page)
            log.info(f"  Running total: {len(rows)} rows")
            r += PAGE_SIZE
            time.sleep(self.delay + random.uniform(0, 0.4))

        if max_rows:
            rows = rows[:max_rows]

        df = pd.DataFrame(rows)
        log.info(f"Scrape complete: {len(df)} rows, {len(df.columns)} columns.")
        log.info(f"Columns returned by Finviz: {list(df.columns)}")
        return df


## Step 3: Data Cleaning & Standardization
This section maps the messy HTML headers from Finviz to our internal taxonomy. It strips out strings (like "B" for Billions and "%" for percentages) and safely coerces them into usable floats for analysis. It also assigns our dynamic Valuation Flags.


In [ ]:

# Map any Finviz header variant → internal column names
COLUMN_ALIASES: dict[str, str] = {
    "no.":              "row_no",
    "#":                "row_no",
    "ticker":           "Ticker",
    "company":          "Company Name",
    "sector":           "GICS Sector",
    "industry":         "GICS Sub-Industry",
    "market cap":       "Market Cap ($M)",      # col 6
    "p/e":              "LTM P/E",              # col 7
    "fwd p/e":          "NTM P/E",              # col 8
    "forward p/e":      "NTM P/E",
    "eps":              "EPS TTM",              # col 16 (EPS ttm)
    "eps (ttm)":        "EPS TTM",
    "eps next q":             "CQ+1 EPS Estimate",
    "eps estimate next quarter": "CQ+1 EPS Estimate",
    "eps est. next q":        "CQ+1 EPS Estimate",  # col 77
    "dividend %":       "Dividend Yield (%)",   # col 14
    "dividend":         "Dividend Yield (%)",
    "div yield":        "Dividend Yield (%)",
    # "price":            "Share Price",          # col 65
    # "volume":           "Volume",               # col 67
    # "change":           "Change (%)",           # col 66 (daily % change)
}


def _parse_market_cap(val) -> float | None:
    """'2.85T' → 2_850_000, '342.10B' → 342_100, '8.50M' → 8.50 (all in $M)."""
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    s = str(val).strip().replace(",", "")
    m = re.match(r"^([\d.]+)([TBMKtbmk]?)$", s)
    if not m:
        return None
    num    = float(m.group(1))
    suffix = m.group(2).upper()
    return num * {"T": 1_000_000, "B": 1_000, "M": 1, "K": 0.001}.get(suffix, 1)

def _parse_pct(val) -> float | None:
    """'2.45%' or '2.45' → 2.45 numeric."""
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    try:
        return float(re.sub(r"[%,\s]", "", str(val)))
    except ValueError:
        return None

def _parse_float(val) -> float | None:
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    try:
        return float(str(val).replace(",", ""))
    except ValueError:
        return None

def standardise(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Rename Finviz columns → internal names, coerce numerics,
    derive computed columns, add Valuation Flag.
    """
    df = df_raw.copy()

    # 1. Rename: case-insensitive match
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower()
        if key in COLUMN_ALIASES:
            rename_map[col] = COLUMN_ALIASES[key]
    df.rename(columns=rename_map, inplace=True)

    # 2. Ensure required columns exist
    for req in [
        "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
        "Market Cap ($M)","LTM P/E","NTM P/E",
        "Dividend Yield (%)","EPS YoY","EPS QoQ"
    ]:
        if req not in df.columns:
            df[req] = None

    # 3. Numeric coercion
    # EPS TTM and CQ+1 EPS from new columns
    df["EPS TTM"]           = df["EPS TTM"].apply(_parse_float)
    df["CQ+1 EPS Estimate"] = df["CQ+1 EPS Estimate"].apply(_parse_float)
    # Market cap, P/E, Div Yield as before
    df["Market Cap ($M)"]    = df["Market Cap ($M)"].apply(_parse_market_cap)
    df["LTM P/E"]            = df["LTM P/E"].apply(_parse_float)
    df["NTM P/E"]            = df["NTM P/E"].apply(_parse_float)
    df["Dividend Yield (%)"] = df["Dividend Yield (%)"].apply(_parse_pct)

    # # Price, Volume, Change
    # df["Share Price"] = df["Share Price"].apply(_parse_float)
    # df["Volume"]      = df["Volume"].apply(_parse_float)
    # df["Change (%)"]  = df["Change (%)"].apply(_parse_pct)


    # 4. Zero / negative P/E → None (exclude from valuation averages)
    for col in ["LTM P/E","NTM P/E"]:
        df.loc[df[col].notna() & (df[col] <= 0), col] = None

    # 5. Derived metrics
    total_mc = df["Market Cap ($M)"].sum(skipna=True)
    df["% of S&P 500 Index"] = (df["Market Cap ($M)"] / total_mc * 100
                                if total_mc else None)

    df["LTM P/E vs Hist Avg (%)"] = df["LTM P/E"].apply(
        lambda x: (x - HIST_AVG_PE) / HIST_AVG_PE * 100 if pd.notna(x) else None
    )
    df["NTM P/E vs Hist Avg (%)"] = df["NTM P/E"].apply(
        lambda x: (x - HIST_AVG_PE) / HIST_AVG_PE * 100 if pd.notna(x) else None
    )

    def _flag(ntm):
        if pd.isna(ntm):            return "N/A"
        if ntm > PE_HIGH_BOUND:     return "Above Avg"
        if ntm < PE_LOW_BOUND:      return "Below Avg"
        return "Within Avg"

    df["Valuation Flag"] = df["NTM P/E"].apply(_flag)

    # 6. Drop helper cols
    df.drop(columns=[c for c in ("row_no","Country") if c in df.columns],
            inplace=True, errors="ignore")

    # 7. Deduplicate by ticker
    df.drop_duplicates(subset=["Ticker"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


## Step 4: Sector Aggregations
Groups the companies by GICS Sector and Sub-Industry, computing the Market-Cap Weighted Averages for P/E to visualize broader market trends.


In [ ]:
def _wavg(grp: pd.DataFrame, val_col: str, wt: str = "Market Cap ($M)") -> float | None:
    v = grp.dropna(subset=[val_col, wt])
    v = v[v[val_col] > 0]
    if v.empty:
        return None
    return (v[val_col] * v[wt]).sum() / v[wt].sum()

def build_summary(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    total_mc = df["Market Cap ($M)"].sum(skipna=True)
    rows = []
    for name, grp in df.groupby(group_col, sort=True):
        mc   = grp["Market Cap ($M)"].sum(skipna=True)
        vn   = grp["NTM P/E"].dropna(); vn = vn[vn > 0]
        vl   = grp["LTM P/E"].dropna(); vl = vl[vl > 0]
        wntm = _wavg(grp, "NTM P/E")
        rows.append({
            group_col:                    name,
            "No. of Companies":           len(grp),
            "% of Index":                 mc / total_mc * 100 if total_mc else None,
            "Weighted Avg LTM P/E":       _wavg(grp, "LTM P/E"),
            "Weighted Avg NTM P/E":       wntm,
            "Weighted Avg EPS TTM": _wavg(grp, "EPS TTM"),
            "Weighted Avg CQ+1 EPS": _wavg(grp, "CQ+1 EPS Estimate"),

            "Weighted Avg Div Yield (%)": _wavg(grp, "Dividend Yield (%)"),
            "Total Market Cap ($M)":      mc,
            "Median LTM P/E":             float(vl.median()) if len(vl) else None,
            "Median NTM P/E":             float(vn.median()) if len(vn) else None,
            "Min NTM P/E":                float(vn.min())    if len(vn) else None,
            "Max NTM P/E":                float(vn.max())    if len(vn) else None,
            "Valuation Flag": (
                "N/A"        if pd.isna(wntm) else
                "Above Avg"  if wntm > PE_HIGH_BOUND else
                "Below Avg"  if wntm < PE_LOW_BOUND else
                "Within Avg"
            ),
        })
    return (pd.DataFrame(rows)
              .sort_values("Weighted Avg NTM P/E", ascending=False, na_position="last")
              .reset_index(drop=True))


## Step 5: Excel Workbook Builder
Transforms the Pandas DataFrames into a formatted, multi-sheet `.xlsx` file using `openpyxl`. Applies thematic colors, conditional formatting for Valuation Flags, and generates a Bar Chart.


In [ ]:

# Palette
NAVY     = "1B2A4A"
WHITE    = "FFFFFF"
ROW_EVEN = "F0F3F7"
ROW_ODD  = "FFFFFF"
R_FILL = "FADBD8"; R_FONT = "922B21"
Y_FILL = "FEF9E7"; Y_FONT = "7D6608"
G_FILL = "D5F5E3"; G_FONT = "1E8449"
N_FILL = "EAECEE"; N_FONT = "717D7E"

def make_labels(as_of: datetime) -> tuple[str, str]:
    """Return (title_text, footer_text) stamped with the given as-of date."""
    d = as_of.strftime("%B %d, %Y")
    title = (
        f"S&P 500 Valuation Analysis — As of {d}  |  "
        "ECM Research  |  Bulge Bracket Bank"
    )
    footer = (
        "Source: Finviz.com Screener | Multpl.com Historical Data | "
        f"Data as of {d}.  "
        "For internal use only. Negative / unavailable P/E multiples excluded.  "
        f"Historical benchmark: {HIST_AVG_PE}x (Scraped S&P 500 Long-Term Median TTM P/E). "
        f"Valuation Bands: Below Avg (<{PE_LOW_BOUND}x) | Within Avg | Above Avg (>{PE_HIGH_BOUND}x)."
    )
    return title, footer


_thin = lambda c="CCCCCC": Side(style="thin", color=c)
DATA_BORDER = Border(left=_thin(), right=_thin(), top=_thin(), bottom=_thin())

def _fill(h): return PatternFill("solid", start_color=h)
def _font(bold=False, size=9, color="000000", italic=False):
    return Font(name="Arial", bold=bold, size=size, color=color, italic=italic)
def _align(h="center", v="center", wrap=False, indent=0):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap, indent=indent)
def _row_fill(i): return _fill(ROW_EVEN if i % 2 == 0 else ROW_ODD)

def _flag_styles(flag):
    return {
        "Above Avg":  (_fill(R_FILL), _font(bold=True, color=R_FONT)),
        "Within Avg": (_fill(Y_FILL), _font(bold=True, color=Y_FONT)),
        "Below Avg":  (_fill(G_FILL), _font(bold=True, color=G_FONT)),
    }.get(flag, (_fill(N_FILL), _font(bold=True, color=N_FONT)))

def _ntm_fill(v):
    if pd.isna(v): return _fill(N_FILL)
    return _fill(R_FILL if v > PE_HIGH_BOUND else G_FILL if v < PE_LOW_BOUND else Y_FILL)

def _write_title(ws, txt, ncols, row=1):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=txt)
    c.font = _font(bold=True, size=12, color=WHITE)
    c.fill = _fill(NAVY)
    c.alignment = _align(h="left", indent=1)
    ws.row_dimensions[row].height = 24

def _write_headers(ws, hdrs, row=2):
    ws.row_dimensions[row].height = 36
    hb = Border(left=_thin("4A5568"), right=_thin("4A5568"),
                top=_thin("4A5568"), bottom=_thin("4A5568"))
    for ci, h in enumerate(hdrs, 1):
        c = ws.cell(row=row, column=ci, value=h)
        c.font = _font(bold=True, color=WHITE, size=9)
        c.fill = _fill(NAVY)
        c.alignment = _align(wrap=True)
        c.border = hb

def _write_footer(ws, txt, ncols, last_row):
    fr = last_row + 2
    ws.merge_cells(start_row=fr, start_column=1, end_row=fr, end_column=ncols)
    c = ws.cell(row=fr, column=1, value=txt)
    c.font = _font(italic=True, size=7, color="888888")
    c.alignment = _align(h="left")

def _col_widths(ws, widths):
    for ci, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

def _apply_fmt(cell, col, val):
    """Apply number formats for Excel."""
    if col in ("Market Cap ($M)", "Total Market Cap ($M)"):
        cell.number_format = '$#,##0'
    elif col in ("Share Price",):
        cell.number_format = '$#,##0.00'
    elif col in ("% of S&P 500 Index","% of Index",
                 "Dividend Yield (%)","Weighted Avg Div Yield (%)"):
        cell.number_format = '0.00%'
        if pd.notna(val): cell.value = val / 100
    elif any(x in col for x in ("P/E","Avg LTM P/E","Avg NTM P/E",
                                "Median LTM","Median NTM","Min NTM","Max NTM")):
        cell.number_format = '0.0"x"'
    elif "Hist Avg" in col:
        cell.number_format = '+0.0%;-0.0%;"-"'
        if pd.notna(val): cell.value = val / 100
    elif "EPS" in col:
        cell.number_format = '0.00'
    elif col == "No. of Companies":
        cell.number_format = '0'


In [27]:
S1_HDRS = [
    "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
    "Market Cap ($M)","% of S&P 500 Index",
    # "Share Price","Change (%)","Volume",
    "LTM P/E","NTM P/E",
    "Dividend Yield (%)",
    "EPS TTM","CQ+1 EPS Estimate",
    "LTM P/E vs Hist Avg (%)","NTM P/E vs Hist Avg (%)",
    "Valuation Flag",
]

S1_COLS = [
    "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
    "Market Cap ($M)","% of S&P 500 Index",
    # "Share Price","Change (%)","Volume",
    "LTM P/E","NTM P/E",
    "Dividend Yield (%)",
    "EPS TTM","CQ+1 EPS Estimate",
    "LTM P/E vs Hist Avg (%)","NTM P/E vs Hist Avg (%)",
    "Valuation Flag",
]


def build_master(wb, df, title_txt, footer_txt):
    ws = wb.active
    ws.title = "S&P500 Master Table"
    _write_title(ws, title_txt, len(S1_HDRS))
    _write_headers(ws, S1_HDRS)
    for di, (_, row) in enumerate(df.iterrows(), 1):
        ri   = di + 2
        flag = row.get("Valuation Flag","N/A")
        bf   = _row_fill(di)
        for ci, col in enumerate(S1_COLS, 1):
            val = row.get(col)
            c   = ws.cell(row=ri, column=ci, value=val)
            c.font   = _font(size=9)
            c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci <= 2 else "center")
            if   col == "NTM P/E":       c.fill = _ntm_fill(row.get("NTM P/E"))
            elif col == "Valuation Flag": c.fill, c.font = _flag_styles(flag)
            else:                         c.fill = bf
            _apply_fmt(c, col, val)
    lr = 2 + len(df)
    ws.freeze_panes = "A3"
    ws.auto_filter.ref = f"A2:{get_column_letter(len(S1_HDRS))}{lr}"
    _write_footer(ws, footer_txt, len(S1_HDRS), lr)
    _col_widths(ws, [8, 32, 24, 36, 14, 11, 8, 8, 12, 10, 10, 18, 18, 13])



In [28]:
S2_HDRS = ["Name","% of Index","# Companies",
           "Wtd Avg LTM P/E","Wtd Avg NTM P/E",
           "Wtd Avg Div Yield (%)","Total Mkt Cap ($M)",
           "Median LTM P/E","Median NTM P/E",
           "Min NTM P/E","Max NTM P/E","Valuation Flag"]

S2_COLS_TEMPLATE = [None,
                    "% of Index","No. of Companies",
                    "Weighted Avg LTM P/E","Weighted Avg NTM P/E","Weighted Avg Div Yield (%)",
                    "Total Market Cap ($M)","Median LTM P/E","Median NTM P/E",
                    "Min NTM P/E","Max NTM P/E","Valuation Flag"]

def build_summary_sheet(wb, sdf, group_col, sheet_name, title_txt, footer_txt, add_chart=False):
    ws = wb.create_sheet(sheet_name)
    _write_title(ws, title_txt, len(S2_HDRS))
    _write_headers(ws, S2_HDRS)
    cols = [group_col] + S2_COLS_TEMPLATE[1:]
    for di, (_, row) in enumerate(sdf.iterrows(), 1):
        ri   = di + 2
        flag = row.get("Valuation Flag","N/A")
        bf   = _row_fill(di)
        for ci, col in enumerate(cols, 1):
            val = row.get(col)
            c   = ws.cell(row=ri, column=ci, value=val)
            c.font   = _font(size=9)
            c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci == 1 else "center")
            if   col == "Weighted Avg NTM P/E": c.fill = _ntm_fill(row.get("Weighted Avg NTM P/E"))
            elif col == "Valuation Flag":        c.fill, c.font = _flag_styles(flag)
            else:                                c.fill = bf
            _apply_fmt(c, col, val)
    lr = 2 + len(sdf)
    ws.freeze_panes = "A3"
    ws.auto_filter.ref = f"A2:{get_column_letter(len(S2_HDRS))}{lr}"
    _write_footer(ws, footer_txt, len(S2_HDRS), lr)
    _col_widths(ws, [36,10,12,14,14,16,18,13,13,11,11,13])
    if add_chart:
        _add_chart(ws, sdf, group_col, lr)

def _add_chart(ws, sdf, group_col, lr):
    cr = lr + 4
    ws.cell(row=cr-1, column=1,
            value="Weighted Avg NTM P/E by Sector vs. Historical Average").font = \
        _font(bold=True, size=11, color=NAVY)
    ws.cell(row=cr, column=1, value="Sector").font = _font(bold=True)
    ws.cell(row=cr, column=2, value="NTM P/E").font = _font(bold=True)
    ws.cell(row=cr, column=3, value=f"Hist Avg ({HIST_AVG_PE}x)").font = _font(bold=True)
    for i, (_, row) in enumerate(sdf.iterrows(), 1):
        rr  = cr + i
        ntm = row.get("Weighted Avg NTM P/E")
        ws.cell(row=rr, column=1, value=row.get(group_col, row.iloc[0]))
        ws.cell(row=rr, column=2, value=round(ntm, 2) if pd.notna(ntm) else 0)
        ws.cell(row=rr, column=3, value=HIST_AVG_PE)
    n     = len(sdf)
    chart = BarChart()
    chart.type="col"; chart.grouping="clustered"
    chart.title        = "Weighted Avg NTM P/E by GICS Sector vs. Historical Average"
    chart.y_axis.title = "NTM P/E (x)"
    chart.x_axis.title = "GICS Sector"
    chart.style=10; chart.width=30; chart.height=16
    chart.add_data(Reference(ws, min_col=2, max_col=3,
                             min_row=cr, max_row=cr+n), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=cr+1, max_row=cr+n))
    chart.series[0].graphicalProperties.solidFill = "2E4057"
    chart.series[1].graphicalProperties.solidFill = "E74C3C"
    ws.add_chart(chart, f"A{cr+2}")


In [29]:
def _section_header(ws, title, color, ncols, row):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=title)
    c.font = _font(bold=True, color=WHITE, size=10)
    c.fill = _fill(color)
    c.alignment = _align()
    ws.row_dimensions[row].height = 22

def _subtable(ws, title, color, hdrs, rows, start_row):
    _section_header(ws, title, color, len(hdrs), start_row)
    hr = start_row + 1
    ws.row_dimensions[hr].height = 28
    hb = Border(left=_thin("4A5568"), right=_thin("4A5568"),
                top=_thin("4A5568"),  bottom=_thin("4A5568"))
    for ci, h in enumerate(hdrs, 1):
        c = ws.cell(row=hr, column=ci, value=h)
        c.font = _font(bold=True, color=WHITE, size=9)
        c.fill = _fill(NAVY); c.alignment = _align(wrap=True); c.border = hb
    for ri_off, rd in enumerate(rows):
        ri = hr + 1 + ri_off
        for ci, val in enumerate(rd, 1):
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = _font(size=9); c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci == 1 else "center")
            c.fill = _row_fill(ri_off + 1)
    return hr + 1 + len(rows)

def _co_rows(subset):
    out = []
    for _, r in subset.iterrows():
        out.append([
            r.get("Ticker",""),
            r.get("Company Name",""),
            r.get("GICS Sector",""),
            f"{r['NTM P/E']:.1f}x",
            f"{r['LTM P/E']:.1f}x" if pd.notna(r.get("LTM P/E")) else "N/A",
            f"{r['Dividend Yield (%)']:.2f}%" if pd.notna(r.get("Dividend Yield (%)")) else "N/A",
            f"${r['Market Cap ($M)']:,.0f}M" if pd.notna(r.get("Market Cap ($M)")) else "N/A",
        ])
    return out

def _sec_rows(subset):
    out = []
    for _, r in subset.iterrows():
        wntm = r.get("Weighted Avg NTM P/E")
        wltm = r.get("Weighted Avg LTM P/E")
        mc   = r.get("Total Market Cap ($M)")
        out.append([
            r.iloc[0],
            f"{wntm:.1f}x" if pd.notna(wntm) else "N/A",
            f"{wltm:.1f}x" if pd.notna(wltm) else "N/A",
            int(r.get("No. of Companies",0)),
            f"${mc:,.0f}M"  if pd.notna(mc)  else "N/A",
        ])
    return out

CO_H  = ["Ticker","Company","GICS Sector","NTM P/E","LTM P/E","Div Yield","Mkt Cap ($M)"]
SEC_H = ["Sub-Industry / Sector","Wtd Avg NTM P/E","Wtd Avg LTM P/E","# Co.","Total Mkt Cap"]

def build_extremes(wb, df, sub_df, title_txt, footer_txt):
    ws  = wb.create_sheet("Extremes Dashboard")
    _write_title(ws, title_txt, 7)
    valid = df[df["NTM P/E"].notna() & (df["NTM P/E"] > 0)]
    vsub  = sub_df[sub_df["Weighted Avg NTM P/E"].notna()]
    cur = 3
    cur = _subtable(
        ws,
        "🔴  TOP 25 MOST EXPENSIVE COMPANIES  —  By NTM P/E  (Potential Over-Enthusiasm)",
        "C0392B", CO_H, _co_rows(valid.nlargest(25,"NTM P/E")), cur
    ) + 2
    cur = _subtable(
        ws,
        "🟢  TOP 25 CHEAPEST COMPANIES  —  By NTM P/E, Excl. Negatives  (Potential Over-Selling)",
        "1E8449", CO_H, _co_rows(valid.nsmallest(25,"NTM P/E")), cur
    ) + 2
    cur = _subtable(
        ws,
        "🔴  TOP 25 MOST EXPENSIVE SUB-SECTORS  —  Wtd Avg NTM P/E",
        "7B241C", SEC_H, _sec_rows(vsub.head(25)), cur
    ) + 2
    cur = _subtable(
        ws,
        "🟢  TOP 25 CHEAPEST SUB-SECTORS  —  Wtd Avg NTM P/E",
        "1A5276", SEC_H, _sec_rows(vsub.nsmallest(25,"Weighted Avg NTM P/E")), cur
    ) + 2
    _write_footer(ws, footer_txt, 7, cur)
    _col_widths(ws, [8,32,26,10,10,12,14])

def build_workbook(df, sub_df, sec_df, as_of: datetime) -> Workbook:
    title_txt, footer_txt = make_labels(as_of)
    wb = Workbook()
    log.info("Sheet 1: S&P500 Master Table")
    build_master(wb, df, title_txt, footer_txt)
    log.info("Sheet 2: Sub-Sector Summary")
    build_summary_sheet(wb, sub_df, "GICS Sub-Industry", "Sub-Sector Summary",
                        title_txt, footer_txt)
    log.info("Sheet 3: Sector Summary + chart")
    build_summary_sheet(wb, sec_df, "GICS Sector", "Sector Summary",
                        title_txt, footer_txt, add_chart=True)
    log.info("Sheet 4: Extremes Dashboard")
    build_extremes(wb, df, sub_df, title_txt, footer_txt)
    return wb


## Step 6: Console Overview & Report Generation
Runs the entire pipeline. 

Scraping Finviz -> Standardizing Data -> Grouping -> Printing Summary -> Saving Excel.


In [30]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — CONSOLE SUMMARY
# ─────────────────────────────────────────────────────────────

def print_summary(df, sub_df, as_of: datetime):
    total    = len(df)
    miss_ntm = int(df["NTM P/E"].isna().sum())
    miss_ltm = int(df["LTM P/E"].isna().sum())
    flags    = df["Valuation Flag"].value_counts()
    total_mc = df["Market Cap ($M)"].sum(skipna=True)

    def wt_pe(pe_col):
        v = df[df[pe_col].notna() & (df[pe_col]>0) & df["Market Cap ($M)"].notna()]
        if v.empty: return float("nan")
        return (v[pe_col]*v["Market Cap ($M)"]).sum() / v["Market Cap ($M)"].sum()

    sp_ntm = wt_pe("NTM P/E"); sp_ltm = wt_pe("LTM P/E")
    vsub   = sub_df[sub_df["Weighted Avg NTM P/E"].notna()]

    print("\n" + "═"*72)
    print("   S&P 500 VALUATION ANALYSIS — CONSOLE SUMMARY")
    print(f"   {as_of:%B %d, %Y}  |  ECM Research")
    print("═"*72)
    print(f"\n  DATA COVERAGE (Finviz S&P 500 screener):")
    print(f"    Companies captured              : {total}")
    print(f"    Missing NTM P/E (no consensus)  : {miss_ntm}  ({miss_ntm/max(total,1)*100:.1f}%)")
    print(f"    Missing LTM P/E (neg. earnings) : {miss_ltm}  ({miss_ltm/max(total,1)*100:.1f}%)")
    print(f"\n  VALUATION FLAGS  (thresholds: <{PE_LOW_BOUND}x = Below, "
          f"{PE_LOW_BOUND}–{PE_HIGH_BOUND}x = Within, >{PE_HIGH_BOUND}x = Above):")
    for f in ("Above Avg","Within Avg","Below Avg","N/A"):
        n   = int(flags.get(f, 0))
        bar = "█" * int(n/max(total,1)*40)
        print(f"    {f:<14}  {n:>3} co.  ({n/max(total,1)*100:5.1f}%)  {bar}")
    print(f"\n  S&P 500 INDEX  (Market-Cap Weighted):")
    print(f"    Wtd Avg NTM P/E : {sp_ntm:.1f}x  (vs {HIST_AVG_PE}x hist avg → {(sp_ntm/HIST_AVG_PE-1)*100:+.1f}%)")
    print(f"    Wtd Avg LTM P/E : {sp_ltm:.1f}x  (vs {HIST_AVG_PE}x hist avg → {(sp_ltm/HIST_AVG_PE-1)*100:+.1f}%)")
    print(f"    Total Mkt Cap   : ${total_mc/1_000_000:,.2f}T")
    print(f"\n  TOP 5 MOST EXPENSIVE SUB-SECTORS:")
    for i, (_, r) in enumerate(vsub.head(5).iterrows(), 1):
        print(f"    {i}. {r.iloc[0]:<44} {r['Weighted Avg NTM P/E']:.1f}x")
    print(f"\n  TOP 5 CHEAPEST SUB-SECTORS:")
    for i, (_, r) in enumerate(vsub.nsmallest(5,'Weighted Avg NTM P/E').iterrows(), 1):
        print(f"    {i}. {r.iloc[0]:<44} {r['Weighted Avg NTM P/E']:.1f}x")
    print("\n" + "═"*72)


In [31]:
from pathlib import Path
from datetime import datetime

# JUPYTER ENTRY POINT (no CLI)
AS_OF_STR   = "2025-04-11"  # or None for today
DEMO_MODE   = False          # True = use built-in sample; False = live Finviz
MAX_ROWS    = None          # optional cap for live scrape

as_of = datetime.strptime(AS_OF_STR, "%Y-%m-%d") if AS_OF_STR else datetime.today()
date_str = as_of.strftime("%Y-%m-%d")
OUTPUT_PATH = Path(f"SP500_Valuation_Analysis_{date_str}.xlsx")

log.info(f"Report as-of date : {as_of:%B %d, %Y}")

if DEMO_MODE:
    log.info("DEMO MODE — using built-in sample.")
    raw = build_demo_df()
    df  = standardise(raw)
else:
    log.info("LIVE MODE — scraping Finviz …")
    raw = FinvizScraper(delay=1.2).fetch_sp500(max_rows=MAX_ROWS)
    df  = standardise(raw)

log.info(f"Clean dataset : {len(df)} companies.")

sub_df = build_summary(df, "GICS Sub-Industry")
sec_df = build_summary(df, "GICS Sector")

wb = build_workbook(df, sub_df, sec_df, as_of)
wb.save(str(OUTPUT_PATH))
print_summary(df, sub_df, as_of)
OUTPUT_PATH


16:12:15  INFO     Report as-of date : April 11, 2025
16:12:15  INFO     LIVE MODE — scraping Finviz …
16:12:15  INFO     Connecting to Finviz S&P 500 screener …
16:12:16  WARNING  Could not read total count — will paginate until empty page.
16:12:16  INFO     Finviz reports 600 S&P 500 constituents.
16:12:16  INFO       Page r=1 → 20 rows
16:12:17  INFO       Fetching rows 21–40 …
16:12:17  INFO       Running total: 40 rows
16:12:19  INFO       Fetching rows 41–60 …
16:12:19  INFO       Running total: 60 rows
16:12:20  INFO       Fetching rows 61–80 …
16:12:21  INFO       Running total: 80 rows
16:12:22  INFO       Fetching rows 81–100 …
16:12:22  INFO       Running total: 100 rows
16:12:24  INFO       Fetching rows 101–120 …
16:12:24  INFO       Running total: 120 rows
16:12:25  INFO       Fetching rows 121–140 …
16:12:26  INFO       Running total: 140 rows
16:12:27  INFO       Fetching rows 141–160 …
16:12:27  INFO       Running total: 160 rows
16:12:28  INFO       Fetching rows 161


════════════════════════════════════════════════════════════════════════
   S&P 500 VALUATION ANALYSIS — CONSOLE SUMMARY
   April 11, 2025  |  ECM Research
════════════════════════════════════════════════════════════════════════

  DATA COVERAGE (Finviz S&P 500 screener):
    Companies captured              : 503
    Missing NTM P/E (no consensus)  : 5  (1.0%)
    Missing LTM P/E (neg. earnings) : 30  (6.0%)

  VALUATION FLAGS  (thresholds: <12.72x = Below, 12.72–17.72x = Within, >17.72x = Above):
    Above Avg       244 co.  ( 48.5%)  ███████████████████
    Within Avg      118 co.  ( 23.5%)  █████████
    Below Avg       136 co.  ( 27.0%)  ██████████
    N/A               5 co.  (  1.0%)  

  S&P 500 INDEX  (Market-Cap Weighted):
    Wtd Avg NTM P/E : 24.7x  (vs 15.22x hist avg → +62.0%)
    Wtd Avg LTM P/E : 42.7x  (vs 15.22x hist avg → +180.5%)
    Total Mkt Cap   : $64.81T

  TOP 5 MOST EXPENSIVE SUB-SECTORS:
    1. Auto Manufacturers                           140.5x
    2. REIT 

WindowsPath('SP500_Valuation_Analysis_2025-04-11.xlsx')